# This is Entire Data for my spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('AzureDE-InterviewPrep')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark

In [ ]:
events = spark.createDataFrame(
    [
        ('{"customer_id":"c1","profile":{"city":"Pune","age":28},"tags":["new","in"],"amount":"100"}',),
        ('{"customer_id":"c2","profile":{"city":"Delhi","age":null},"tags":["vip"],"amount":null}',),
        ('{"customer_id":"c3","profile":{"city":"Pune","age":35},"tags":[],"amount":"250.5"}',),
    ],
    ['raw_json'],
)
events.show(truncate=False)

+------------------------------------------------------------------------------------------+
|raw_json                                                                                  |
+------------------------------------------------------------------------------------------+
|{"customer_id":"c1","profile":{"city":"Pune","age":28},"tags":["new","in"],"amount":"100"}|
|{"customer_id":"c2","profile":{"city":"Delhi","age":null},"tags":["vip"],"amount":null}   |
|{"customer_id":"c3","profile":{"city":"Pune","age":35},"tags":[],"amount":"250.5"}        |
+------------------------------------------------------------------------------------------+



# Tasks
- 1. Parse JSON with explicit schema.
- 2. Flatten city/age; cast amount; fill null amount with 0.
- 3. explode_outer tags.
- 4. Why explicit schema over inferSchema in production?

In [ ]:
# Parse JSON with explicit schema.

json_schema = T.StructType([
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("profile", T.StructType([
        T.StructField("city", T.StringType(), True),
        T.StructField("age", T.IntegerType(), True)
    ]), True),
    T.StructField("tags", T.ArrayType(T.StringType()), True),
    T.StructField("amount", T.StringType(), True)
])
parsed_df = events.withColumn("parsed", F.from_json(F.col("raw_json"), json_schema))
parsed_df.select("parsed").show(truncate=False)

+--------------------------------+
|parsed                          |
+--------------------------------+
|{c1, {Pune, 28}, [new, in], 100}|
|{c2, {Delhi, NULL}, [vip], NULL}|
|{c3, {Pune, 35}, [], 250.5}     |
+--------------------------------+



In [ ]:
# Flatten city/age; cast amount; fill null amount with 0.

flattened_df = (
    parsed_df
    .select(
        F.col("parsed.customer_id").alias("customer_id"),
        F.col("parsed.profile.city").alias("city"),
        F.col("parsed.profile.age").alias("age"),
        F.col("parsed.tags").alias("tags"),
        F.col("parsed.amount").cast(T.DoubleType()).alias("amount")
    )
    .fillna({"amount": 0.0})
)

flattened_df.show(truncate=False)

+-----------+-----+----+---------+------+
|customer_id|city |age |tags     |amount|
+-----------+-----+----+---------+------+
|c1         |Pune |28  |[new, in]|100.0 |
|c2         |Delhi|NULL|[vip]    |0.0   |
|c3         |Pune |35  |[]       |250.5 |
+-----------+-----+----+---------+------+



In [ ]:
# explode_outer tags.

exploded_df = flattened_df.select(
    "customer_id",
    "city",
    "age",
    "amount",
    F.explode_outer("tags").alias("tag")
)

exploded_df.show(truncate=False)

+-----------+-----+----+------+----+
|customer_id|city |age |amount|tag |
+-----------+-----+----+------+----+
|c1         |Pune |28  |100.0 |new |
|c1         |Pune |28  |100.0 |in  |
|c2         |Delhi|NULL|0.0   |vip |
|c3         |Pune |35  |250.5 |NULL|
+-----------+-----+----+------+----+



Task Definition: Explain why hardcoded/explicit schemas are required for enterprise production pipelines compared to schema inference.

- Performance Overhead: inferSchema requires a full extra scan over the source dataset just to determine column types. In large production datasets (TB/PB scale), this doubles reading time and causes immense IO/CPU waste.

- Pipeline Determinism & Schema Drift: If source data changes unexpectedly (e.g., a numeric field arrives as a string or a missing field drops out), inferSchema can lead to unpredictable schema changes downstream, breaking ETL tasks. Explicit schemas force strict contract adherence.

- Graceful Handling of Corrupt Records: With an explicit schema, malformed JSON records that fail schema parsing can be caught and isolated using PERMISSIVE mode or columnNameOfCorruptRecord instead of silently failing or changing output types.

- Compile-Time Planning: Explicit schemas allow Spark's Catalyst Optimizer to immediately generate optimized execution plans without executing expensive metadata inspection jobs.